# 05 — Current-cycle queue CSV exploration and parser closure

## Purpose

This notebook characterizes the dated ALMA current-cycle duplication-check CSV
before production parsing. It keeps the physical file layout and all 79 operational
columns as raw evidence, then derives typed diagnostics in separate tables.

The final closure pass answers four parser-blocking questions:

1. Can repeated rows be represented as observed spatial–spectral associations
   without inventing a Cartesian product?
2. Which offset tolerance separates exported floating-point noise from real mosaic
   geometry?
3. Does every regular-row `Ref.Frequency` lie inside a derived sky-frequency SPW
   interval?
4. What do `Req.Sensitivity`, `Ref.Freq.Width`, and the SPS fields actually support?

## Non-negotiable evidence rules

- `queue_raw` remains an unmodified string table.
- A raw-row identity is `(snapshot_sha256, physical_line_number)`; a content hash is
  only a fingerprint because exact duplicate rows exist.
- SPW attributes are aligned only by the same numbered slot.
- Only associations present in source rows may be reconstructed. Missing
  spatial–spectral combinations are never synthesized.
- Raw units and conflicting declarations are preserved next to any normalized value.
- A derived relation is labelled as diagnostic evidence, not promoted to an
  authoritative observing mode or policy verdict.

The pinned input snapshot has SHA-256
`8657108b59295c62d3f1f6635bf3571404f5d43bc5800c4a2e7ea3ba51a111b5`.

In [1]:
from pathlib import Path
from hashlib import sha256
from datetime import datetime, timezone
import csv
import io
import os
import re

import numpy as np
import pandas as pd


EXPECTED_SHA256 = (
    "8657108b59295c62d3f1f6635bf3571404f5d43bc5800c4a2e7ea3ba51a111b5"
)

CSV_URL = (
    "https://almascience.eso.org/"
    "documents-and-tools/latest/duplication-check-csv"
)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


csv_path_override = os.environ.get("ALMA_QUEUE_CSV_PATH")

if csv_path_override:
    CSV_PATH = Path(csv_path_override).expanduser().resolve()
    REPO_ROOT = None
else:
    REPO_ROOT = find_repo_root(Path.cwd())
    CSV_PATH = (
        REPO_ROOT
        / "data"
        / "raw"
        / "projects_in_queue_cycle13_20260901.csv"
    )

raw_bytes = CSV_PATH.read_bytes()
raw_text = raw_bytes.decode("utf-8-sig")
checksum = sha256(raw_bytes).hexdigest()

captured_at = datetime.fromtimestamp(
    CSV_PATH.stat().st_mtime,
    tz=timezone.utc,
)

display_path = (
    CSV_PATH.relative_to(REPO_ROOT)
    if REPO_ROOT is not None
    else Path(CSV_PATH.name)
)

print("Source URL:", CSV_URL)
print("Path:", display_path)
print("Bytes:", len(raw_bytes))
print("SHA-256:", checksum)
print("Local file mtime (not source provenance):", captured_at.isoformat())

assert checksum == EXPECTED_SHA256


Source URL: https://almascience.eso.org/documents-and-tools/latest/duplication-check-csv
Path: duplication-check-csv (1).csv
Bytes: 1448241
SHA-256: 8657108b59295c62d3f1f6635bf3571404f5d43bc5800c4a2e7ea3ba51a111b5
Local file mtime (not source provenance): 2026-09-01T09:57:27.394873+00:00


## 1. File layout and raw evidence

The download is not a single ordinary CSV table. It contains a description, an
embedded three-column field dictionary, a blank separator, a 79-column operational
header, a separate 79-value unit row, and the data records. These layers are parsed
explicitly so that source metadata is not mistaken for observations.


In [2]:
rows = list(csv.reader(io.StringIO(raw_text)))

DESCRIPTION_INDEX = 0
DICTIONARY_HEADER_INDEX = 2
DICTIONARY_START_INDEX = 3
DICTIONARY_END_INDEX = 38
TABLE_HEADER_INDEX = 39
TABLE_UNITS_INDEX = 40
TABLE_DATA_START_INDEX = 41

description = rows[DESCRIPTION_INDEX]
dictionary_header = rows[DICTIONARY_HEADER_INDEX]
dictionary_rows = rows[DICTIONARY_START_INDEX:DICTIONARY_END_INDEX]
column_names = rows[TABLE_HEADER_INDEX]
column_units = rows[TABLE_UNITS_INDEX]
data_rows = rows[TABLE_DATA_START_INDEX:]

print("Description:", description[0])
print("Physical rows:", len(rows))
print("Dictionary rows:", len(dictionary_rows))
print("Operational columns:", len(column_names))
print("Data rows:", len(data_rows))

assert len(rows) == 3241
assert dictionary_header == ["Column Heading", "Units", "Description"]
assert len(dictionary_rows) == 35
assert len(column_names) == 79
assert len(column_units) == 79
assert len(data_rows) == 3200
assert all(len(row) == 79 for row in data_rows)


Description: This spreadsheet lists the metadata for ongoing observations for Grade A projects that are in the observing queue as of 2026 March 3.
Physical rows: 3241
Dictionary rows: 35
Operational columns: 79
Data rows: 3200


In [3]:
queue_raw = pd.DataFrame(data_rows, columns=column_names)

queue_dictionary = pd.DataFrame(
    dictionary_rows,
    columns=dictionary_header,
)

queue_units = pd.DataFrame(
    {
        "column_name": column_names,
        "raw_unit": column_units,
        "position": range(len(column_names)),
    }
)

raw_shape = queue_raw.shape
raw_columns = tuple(queue_raw.columns)
raw_content_hash = sha256(
    "\n".join(
        "\x1f".join(row)
        for row in data_rows
    ).encode("utf-8")
).hexdigest()

assert raw_shape == (3200, 79)
assert not queue_raw.columns.duplicated().any()

display(queue_dictionary)
display(queue_units)
display(queue_raw.head())

print("Raw content hash:", raw_content_hash)


Raw content hash: c9aaca65750d567c3760319db6484d9f6e2e64661969a758f47c6342ffd88dad


,Column Heading,Units,Description
0,Project Code,string,ALMA Project code.
1,Target Name,string,Name of the science target.
2,RA,[deg],"Target right ascension, in decimal degrees (0 ..."
3,Dec,[deg],"Target declination, in decimal degress (0 for ..."
4,RA_HMS,[h:m:s],"Target right ascension, in Hr,Min,Sec."
5,Dec_DMS,[d:m:s],"Target declination, in Deg, Min, Sec."
6,Long Offset,"[""]",Right Ascension/galactic-longitude offset in a...
7,Lat Offset,"[""]",Declination/galactic-latitude offset in arcsec...
8,Velocity,[km/s],Target velocity
9,Vel. Frame,string,Target velocity reference frame.


,column_name,raw_unit,position
0,Project Code,,0
1,Target Name,,1
2,RA,[deg],2
3,Dec,[deg],3
4,RA_HMS,,4
...,...,...,...
74,Spec.Res. SPW 12,[MHz],74
75,Spec.Res. SPW 13,[MHz],75
76,Spec.Res. SPW 14,[MHz],76
77,Spec.Res. SPW 15,[MHz],77


,Project Code,Target Name,RA,Dec,RA_HMS,Dec_DMS,Long Offset,Lat Offset,Velocity,Vel. Frame,Vel. Convention,Mosaic,Mos. Length,Mos. Width,Mos. PA,Mos. Spacing,Mos. Coord.,Band,Req. Ang. Res.,Req. LAS,Use 7-m?,Use TP?,Polarization,Ref.Frequency,Ref.Freq.Width,Req.Sensitivity,Is Sky Freq?,SPS Start Freq.,SPS End Freq.,SPS Bandwidth,SPS Spec. Res.,Freq SPW 1,Freq SPW 2,Freq SPW 3,Freq SPW 4,Freq SPW 5,Freq SPW 6,Freq SPW 7,Freq SPW 8,Freq SPW 9,Freq SPW 10,Freq SPW 11,Freq SPW 12,Freq SPW 13,Freq SPW 14,Freq SPW 15,Freq SPW 16,Bandwidth SPW 1,Bandwidth SPW 2,Bandwidth SPW 3,Bandwidth SPW 4,Bandwidth SPW 5,Bandwidth SPW 6,Bandwidth SPW 7,Bandwidth SPW 8,Bandwidth SPW 9,Bandwidth SPW 10,Bandwidth SPW 11,Bandwidth SPW 12,Bandwidth SPW 13,Bandwidth SPW 14,Bandwidth SPW 15,Bandwidth SPW 16,Spec.Res. SPW 1,Spec.Res. SPW 2,Spec.Res. SPW 3,Spec.Res. SPW 4,Spec.Res. SPW 5,Spec.Res. SPW 6,Spec.Res. SPW 7,Spec.Res. SPW 8,Spec.Res. SPW 9,Spec.Res. SPW 10,Spec.Res. SPW 11,Spec.Res. SPW 12,Spec.Res. SPW 13,Spec.Res. SPW 14,Spec.Res. SPW 15,Spec.Res. SPW 16
0,2024.1.00750.S,NGC6240,253.24541666666667,2.4010972222222224,16h52m58.90s,2d24m04.0s,0.0,-4.7231237019733225e-11,7338.919371839999,lsrk,OPTICAL,,0.0,0.0,0.0,0.0,,ALMA_RB_07,0.4,2.0,False,False,FULL,338.5,7500.0,0.008,True,,,,,350.4999999999,348.5000000001,338.4999999999,336.5000000001,,,,,,,,,,,,,1875.0,1875.0,1875.0,1875.0,,,,,,,,,,,,,15.6240234375,15.6240234375,15.6240234375,15.6240234375,,,,,,,,,,,,
1,2024.1.01166.S,"Tsuchinshan-ATLAS_C2023_A3 Epoch 2, band 1",252.22779233333335,2.784745763888888,16h48m54.67s,2d47m05.1s,0.0,0.0,0.0,topo,RADIO,,0.0,0.0,0.0,0.0,,ALMA_RB_01,29.535755499096968,10.0,True,False,DOUBLE,43.188,125.0,0.1,True,,,,,43.188,41.188,39.125,37.188,,,,,,,,,,,,,1875.0,1875.0,1875.0,1875.0,,,,,,,,,,,,,31.25,31.25,31.25,31.25,,,,,,,,,,,,
2,2024.1.01166.S,"Tsuchinshan-ATLAS_C2023_A3 Copy of Epoch 2, ba...",252.22779233333335,2.784745763888888,16h48m54.67s,2d47m05.1s,0.0,0.0,0.0,topo,RADIO,,0.0,0.0,0.0,0.0,,ALMA_RB_07,3.891437645179366,10.0,True,False,DOUBLE,343.7,63.064453125,1.1,False,,,,,345.79599,354.505473,345.50889060915586,343.7,356.7,355.5091479350359,,,,,,,,,,,125.0,125.0,125.0,1875.0,1875.0,125.0,,,,,,,,,,,0.14111328125,0.14111328125,0.14111328125,31.25,31.25,0.14111328125,,,,,,,,,,
3,2024.1.01273.S,alf_sco,247.35191542,-26.43200261,16h29m24.46s,-26d25m55.2s,0.0,0.0,-3.5,hel,RELATIVISTIC,,0.0,0.0,0.0,0.0,,ALMA_RB_10,0.006,0.055,False,False,DOUBLE,870.91016759332,4.515625,1.0,False,,,,,869.0,870.9,850.9,856.85,,,,,,,,,,,,,1875.0,1875.0,1875.0,1875.0,,,,,,,,,,,,,1.12890625,1.12890625,1.12890625,1.12890625,,,,,,,,,,,,
4,2024.1.01273.S,alf_sco,247.35191542,-26.43200261,16h29m24.46s,-26d25m55.2s,0.0,0.0,-3.5,hel,RELATIVISTIC,,0.0,0.0,0.0,0.0,,ALMA_RB_09,0.008,0.07,False,False,DOUBLE,631.3073703084888,10.5290735884437,4.0,False,,,,,647.2,649.25,643.6,631.3,,,,,,,,,,,,,1875.0,1875.0,1875.0,1875.0,,,,,,,,,,,,,1.12890625,1.12890625,1.12890625,1.12890625,,,,,,,,,,,,


## 2. Operational spectral-window schema

The operational table reserves 16 indexed SPW slots. Each slot has three fields that
are aligned only by their shared slot number: central frequency, bandwidth, and
spectral resolution. This is a wide-table schema inventory, not a Cartesian product.

For example, `Freq SPW 4`, `Bandwidth SPW 4`, and `Spec.Res. SPW 4` belong to one
indexed slot. Values from different slot numbers are never combined. The dictionary
uses `Spec.Res SPW [N]`, while the operational header uses `Spec.Res. SPW N`; raw names
are preserved and mapped explicitly.


In [4]:
SPW_PATTERNS = {
    "frequency": re.compile(r"^Freq SPW (?P<number>\d+)$"),
    "bandwidth": re.compile(r"^Bandwidth SPW (?P<number>\d+)$"),
    "spectral_resolution": re.compile(
        r"^Spec\.Res\. SPW (?P<number>\d+)$"
    ),
}

spw_groups = {number: {} for number in range(1, 17)}
matched_spw_columns = []

for column in queue_raw.columns:
    for quantity, pattern in SPW_PATTERNS.items():
        match = pattern.fullmatch(column)
        if match is None:
            continue

        spw_number = int(match.group("number"))
        assert 1 <= spw_number <= 16
        assert quantity not in spw_groups[spw_number]

        spw_groups[spw_number][quantity] = column
        matched_spw_columns.append(column)
        break

expected_quantities = {
    "frequency",
    "bandwidth",
    "spectral_resolution",
}

for spw_number, fields in spw_groups.items():
    assert set(fields) == expected_quantities, (spw_number, fields)

# Schema inventory only: 16 indexed slots with three aligned fields each.
# This assertion does not construct a Cartesian product.
assert len(matched_spw_columns) == 48
assert len(set(matched_spw_columns)) == 48

print("Reserved SPW slots:", list(spw_groups))
print("SPW-related source columns:", len(matched_spw_columns))
print("SPW 1 mapping:", spw_groups[1])
print("SPW 16 mapping:", spw_groups[16])


Reserved SPW slots: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
SPW-related source columns: 48
SPW 1 mapping: {'frequency': 'Freq SPW 1', 'bandwidth': 'Bandwidth SPW 1', 'spectral_resolution': 'Spec.Res. SPW 1'}
SPW 16 mapping: {'frequency': 'Freq SPW 16', 'bandwidth': 'Bandwidth SPW 16', 'spectral_resolution': 'Spec.Res. SPW 16'}


In [5]:
spw_slot_records = []
spw_presence = {}

for spw_number, fields in spw_groups.items():
    slot_columns = [
        fields["frequency"],
        fields["bandwidth"],
        fields["spectral_resolution"],
    ]
    slot_values = queue_raw[slot_columns].apply(
        lambda column: column.astype(str).str.strip()
    )
    populated_count = slot_values.ne("").sum(axis=1)

    spw_presence[spw_number] = slot_values[
        fields["frequency"]
    ].ne("")

    spw_slot_records.append(
        {
            "spw_number": spw_number,
            "complete_rows": int(populated_count.eq(3).sum()),
            "partial_rows": int(populated_count.between(1, 2).sum()),
            "empty_rows": int(populated_count.eq(0).sum()),
        }
    )

spw_slot_summary = pd.DataFrame(spw_slot_records)
spw_presence = pd.DataFrame(spw_presence)

derived_spw_count = (
    spw_presence.sum(axis=1)
    .astype("int64")
    .rename("derived_spw_count")
)

spw_count_summary = (
    derived_spw_count.value_counts()
    .sort_index()
    .rename_axis("spw_count")
    .reset_index(name="row_count")
)

display(spw_slot_summary)
display(spw_count_summary)

assert spw_slot_summary["partial_rows"].sum() == 0
assert derived_spw_count.max() == 7
assert queue_raw.shape == raw_shape
assert tuple(queue_raw.columns) == raw_columns


,spw_number,complete_rows,partial_rows,empty_rows
0,1,3199,0,1
1,2,3199,0,1
2,3,3199,0,1
3,4,3197,0,3
4,5,1731,0,1469
5,6,1655,0,1545
6,7,36,0,3164
7,8,0,0,3200
8,9,0,0,3200
9,10,0,0,3200


,spw_count,row_count
0,0,1
1,3,2
2,4,1466
3,5,76
4,6,1619
5,7,36


In [6]:
non_contiguous_rows = []

for row_index, presence in spw_presence.iterrows():
    populated_slots = [
        number
        for number, is_present in presence.items()
        if is_present
    ]

    if populated_slots:
        expected_slots = list(range(1, max(populated_slots) + 1))
        if populated_slots != expected_slots:
            non_contiguous_rows.append(
                {
                    "row_index": row_index,
                    "populated_slots": populated_slots,
                }
            )

print("Rows with partial SPW triples:", int(spw_slot_summary["partial_rows"].sum()))
print("Rows with non-contiguous SPW slots:", len(non_contiguous_rows))

assert not non_contiguous_rows


Rows with partial SPW triples: 0
Rows with non-contiguous SPW slots: 0


### Current-snapshot SPW finding

The schema reserves 16 numbered slots, but the 2026-09-01 snapshot uses at most seven.
Every populated slot contains all three aligned fields, and populated slots are
contiguous from slot 1. These are observed properties of this dated snapshot, not
permanent assumptions for a future production parser. Partial and non-contiguous
future rows must be reported rather than silently repaired.


In [7]:
# Create explicit row provenance and a long SPW evidence table without mutating
# queue_raw and without combining fields across slot numbers.
source_line_numbers = pd.Series(
    range(
        TABLE_DATA_START_INDEX + 1,
        TABLE_DATA_START_INDEX + 1 + len(queue_raw),
    ),
    index=queue_raw.index,
    name="source_line_number",
)

source_row_hashes = queue_raw.apply(
    lambda row: sha256(
        "\x1f".join(row.astype(str)).encode("utf-8")
    ).hexdigest(),
    axis=1,
).rename("source_row_hash")

queue_spw_records = []

for row_index, row in queue_raw.iterrows():
    for spw_number, fields in spw_groups.items():
        raw_values = {
            quantity: str(row[column]).strip()
            for quantity, column in fields.items()
        }

        if not any(raw_values.values()):
            continue

        assert all(raw_values.values())

        queue_spw_records.append(
            {
                "source_row_index": int(row_index),
                "source_line_number": int(source_line_numbers[row_index]),
                "source_row_hash": source_row_hashes[row_index],
                "project_code": row["Project Code"],
                "target_name": row["Target Name"],
                "band": row["Band"],
                "spw_number": spw_number,
                "frequency_raw": raw_values["frequency"],
                "bandwidth_raw": raw_values["bandwidth"],
                "spectral_resolution_raw": raw_values[
                    "spectral_resolution"
                ],
                "is_sky_frequency_raw": row["Is Sky Freq?"],
                "velocity_raw": row["Velocity"],
                "velocity_frame_raw": row["Vel. Frame"],
                "velocity_convention_raw": row["Vel. Convention"],
            }
        )

queue_spw_long = pd.DataFrame(queue_spw_records)

queue_spw_long["frequency_ghz"] = pd.to_numeric(
    queue_spw_long["frequency_raw"],
    errors="raise",
)
queue_spw_long["bandwidth_mhz"] = pd.to_numeric(
    queue_spw_long["bandwidth_raw"],
    errors="raise",
)
queue_spw_long["spectral_resolution_mhz"] = pd.to_numeric(
    queue_spw_long["spectral_resolution_raw"],
    errors="raise",
)

print("Long SPW evidence rows:", len(queue_spw_long))
display(queue_spw_long.head())

assert len(queue_spw_long) == int(derived_spw_count.sum())
assert len(queue_spw_long) == 16216
assert queue_raw.shape == raw_shape


Long SPW evidence rows: 16216


,source_row_index,source_line_number,source_row_hash,project_code,target_name,band,spw_number,frequency_raw,bandwidth_raw,spectral_resolution_raw,is_sky_frequency_raw,velocity_raw,velocity_frame_raw,velocity_convention_raw,frequency_ghz,bandwidth_mhz,spectral_resolution_mhz
0,0,42,06cf1ea1fff97fa15823c890308b6a01676723af00fcd1...,2024.1.00750.S,NGC6240,ALMA_RB_07,1,350.4999999999,1875.0,15.6240234375,True,7338.919371839999,lsrk,OPTICAL,350.500,1875.0,15.624023
1,0,42,06cf1ea1fff97fa15823c890308b6a01676723af00fcd1...,2024.1.00750.S,NGC6240,ALMA_RB_07,2,348.5000000001,1875.0,15.6240234375,True,7338.919371839999,lsrk,OPTICAL,348.500,1875.0,15.624023
2,0,42,06cf1ea1fff97fa15823c890308b6a01676723af00fcd1...,2024.1.00750.S,NGC6240,ALMA_RB_07,3,338.4999999999,1875.0,15.6240234375,True,7338.919371839999,lsrk,OPTICAL,338.500,1875.0,15.624023
3,0,42,06cf1ea1fff97fa15823c890308b6a01676723af00fcd1...,2024.1.00750.S,NGC6240,ALMA_RB_07,4,336.5000000001,1875.0,15.6240234375,True,7338.919371839999,lsrk,OPTICAL,336.500,1875.0,15.624023
4,1,43,2b0db8ef57df100978b29d7d8afb2c86bdcf5a40fe23a4...,2024.1.01166.S,"Tsuchinshan-ATLAS_C2023_A3 Epoch 2, band 1",ALMA_RB_01,1,43.188,1875.0,31.25,True,0.0,topo,RADIO,43.188,1875.0,31.250000


## 3. Regular SPW versus spectral-scan representation

The SPS fields form a separate representation. The following census tests whether a
row contains regular indexed SPWs, SPS fields, both, neither, or a partial SPS record.
SPS is not expanded into synthetic ordinary SPWs because the CSV does not yet provide
an explicit reconstruction rule for scan-window centers or spacing.


In [8]:
sps_columns = [
    "SPS Start Freq.",
    "SPS End Freq.",
    "SPS Bandwidth",
    "SPS Spec. Res.",
]

sps_values = queue_raw[sps_columns].apply(
    lambda column: column.astype(str).str.strip()
)
sps_populated_count = sps_values.ne("").sum(axis=1)

regular_spw_mask = derived_spw_count.gt(0)
spectral_scan_mask = sps_populated_count.gt(0)

representation_summary = pd.Series(
    {
        "regular_spw_only": int((regular_spw_mask & ~spectral_scan_mask).sum()),
        "spectral_scan_only": int((~regular_spw_mask & spectral_scan_mask).sum()),
        "both": int((regular_spw_mask & spectral_scan_mask).sum()),
        "neither": int((~regular_spw_mask & ~spectral_scan_mask).sum()),
        "partial_sps": int(sps_populated_count.between(1, 3).sum()),
    }
)

sps_examples = queue_raw.loc[
    spectral_scan_mask,
    [
        "Project Code",
        "Target Name",
        "Band",
        "Is Sky Freq?",
        *sps_columns,
    ],
].copy()
sps_examples["derived_spw_count"] = derived_spw_count.loc[
    spectral_scan_mask
]
sps_examples["source_line_number"] = source_line_numbers.loc[
    spectral_scan_mask
]
sps_examples["source_row_hash"] = source_row_hashes.loc[
    spectral_scan_mask
]

display(representation_summary)
display(sps_examples)

assert representation_summary[
    ["regular_spw_only", "spectral_scan_only", "both", "neither"]
].sum() == len(queue_raw)
assert representation_summary["partial_sps"] == 0
assert representation_summary["both"] == 0
assert representation_summary["neither"] == 0


regular_spw_only      3199
spectral_scan_only       1
both                     0
neither                  0
partial_sps              0
dtype: int64

,Project Code,Target Name,Band,Is Sky Freq?,SPS Start Freq.,SPS End Freq.,SPS Bandwidth,SPS Spec. Res.,derived_spw_count,source_line_number,source_row_hash
33,2025.1.00299.S,HBC_687,ALMA_RB_06,True,261.5,268.7,1000.0,0.000564453125,0,75,f1224d710092d4a2c97c22f41320156debfee249ce38a3...


In [9]:
# Preserve and compare the conflicting SPS Bandwidth unit declarations.
dictionary_sps_bandwidth_unit = queue_dictionary.loc[
    queue_dictionary["Column Heading"].eq("SPS Bandwidth"),
    "Units",
].iloc[0]

unit_row_sps_bandwidth_unit = queue_units.loc[
    queue_units["column_name"].eq("SPS Bandwidth"),
    "raw_unit",
].iloc[0]

sps_row = queue_raw.loc[spectral_scan_mask].iloc[0]
sps_start_ghz = float(sps_row["SPS Start Freq."])
sps_end_ghz = float(sps_row["SPS End Freq."])
sps_bandwidth_raw = float(sps_row["SPS Bandwidth"])
sps_scan_span_ghz = sps_end_ghz - sps_start_ghz

sps_unit_evidence = pd.Series(
    {
        "dictionary_unit": dictionary_sps_bandwidth_unit,
        "operational_unit_row": unit_row_sps_bandwidth_unit,
        "scan_span_ghz": sps_scan_span_ghz,
        "raw_bandwidth_value": sps_bandwidth_raw,
        "bandwidth_if_dictionary_unit_ghz": sps_bandwidth_raw / 1000.0,
        "bandwidth_if_unit_row_ghz": sps_bandwidth_raw,
        "dictionary_interpretation_ratio_to_span": (
            (sps_bandwidth_raw / 1000.0) / sps_scan_span_ghz
        ),
        "unit_row_interpretation_ratio_to_span": (
            sps_bandwidth_raw / sps_scan_span_ghz
        ),
    }
)

display(sps_unit_evidence)

assert dictionary_sps_bandwidth_unit == "[MHz]"
assert unit_row_sps_bandwidth_unit == "[GHz]"


dictionary_unit                                 [MHz]
operational_unit_row                            [GHz]
scan_span_ghz                                     7.2
raw_bandwidth_value                            1000.0
bandwidth_if_dictionary_unit_ghz                  1.0
bandwidth_if_unit_row_ghz                      1000.0
dictionary_interpretation_ratio_to_span      0.138889
unit_row_interpretation_ratio_to_span      138.888889
dtype: object

### SPS unit interpretation

The field dictionary declares `SPS Bandwidth` in MHz, while the operational unit row
declares GHz. The only SPS record spans 7.2 GHz and has a raw bandwidth value of
1000. Interpreting the value as 1000 MHz gives 1 GHz per scan window; interpreting it
as 1000 GHz makes one window far wider than the complete scan. Internal numerical
consistency therefore strongly supports the dictionary's MHz interpretation.

This is evidence, not silent correction. Both declarations and the raw value remain
preserved. Production normalization should record the selected unit source and should
seek confirmation from ALMA documentation or the supervisor.


## 4. Missing values, categorical values, and schema drift

Empty strings, textual sentinels, and numeric zero are counted separately. Zero is not
treated as missing because it can be a valid coordinate offset, velocity, or documented
solar-system placeholder. Categorical values are inspected before normalization.


In [10]:
missing_records = []

for column in queue_raw.columns:
    values = queue_raw[column].astype(str).str.strip()
    upper_values = values.str.upper()

    missing_records.append(
        {
            "column_name": column,
            "empty": int(values.eq("").sum()),
            "N/A": int(upper_values.eq("N/A").sum()),
            "NULL": int(upper_values.eq("NULL").sum()),
            "NONE": int(upper_values.eq("NONE").sum()),
            "zero": int(values.isin(["0", "0.0"]).sum()),
            "unique_nonempty": int(values[values.ne("")].nunique()),
        }
    )

missing_summary = pd.DataFrame(missing_records)

display(
    missing_summary.sort_values(
        ["empty", "N/A", "NULL", "NONE"],
        ascending=False,
    ).head(35)
)

assert missing_summary[["N/A", "NULL", "NONE"]].to_numpy().sum() == 0


,column_name,empty,N/A,NULL,NONE,zero,unique_nonempty
38,Freq SPW 8,3200,0,0,0,0,0
39,Freq SPW 9,3200,0,0,0,0,0
40,Freq SPW 10,3200,0,0,0,0,0
41,Freq SPW 11,3200,0,0,0,0,0
42,Freq SPW 12,3200,0,0,0,0,0
43,Freq SPW 13,3200,0,0,0,0,0
44,Freq SPW 14,3200,0,0,0,0,0
45,Freq SPW 15,3200,0,0,0,0,0
46,Freq SPW 16,3200,0,0,0,0,0
54,Bandwidth SPW 8,3200,0,0,0,0,0


In [11]:
categorical_columns = [
    "Vel. Frame",
    "Vel. Convention",
    "Mosaic",
    "Mos. Coord.",
    "Band",
    "Use 7-m?",
    "Use TP?",
    "Polarization",
    "Is Sky Freq?",
]

categorical_records = []

for column in categorical_columns:
    counts = (
        queue_raw[column]
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
    )
    for raw_value, row_count in counts.items():
        categorical_records.append(
            {
                "column_name": column,
                "raw_value": raw_value if raw_value else "<blank>",
                "row_count": int(row_count),
            }
        )

categorical_summary = pd.DataFrame(categorical_records)
display(categorical_summary)

assert set(queue_raw["Is Sky Freq?"]) == {"True", "False"}
assert set(queue_raw["Use 7-m?"]) == {"True", "False"}
assert set(queue_raw["Use TP?"]) == {"True", "False"}
assert set(queue_raw["Mosaic"]) == {"Custom", "Rectangle", ""}


,column_name,raw_value,row_count
0,Vel. Frame,lsrk,3085
1,Vel. Frame,hel,109
2,Vel. Frame,topo,6
3,Vel. Convention,RADIO,3077
4,Vel. Convention,RELATIVISTIC,110
5,Vel. Convention,OPTICAL,13
6,Mosaic,Custom,2940
7,Mosaic,Rectangle,140
8,Mosaic,<blank>,120
9,Mos. Coord.,<blank>,3060


In [12]:
schema_discrepancies = pd.DataFrame(
    [
        {
            "topic": "SPS Bandwidth unit",
            "dictionary": "[MHz]",
            "operational_table": "[GHz]",
            "status": "conflict; preserve both",
        },
        {
            "topic": "Velocity unit spelling",
            "dictionary": "[km/s]",
            "operational_table": "[kms/s]",
            "status": "lexical discrepancy",
        },
        {
            "topic": "Mosaic datatype",
            "dictionary": "[boolean]",
            "operational_table": "Custom / Rectangle / blank",
            "status": "categorical, not boolean",
        },
        {
            "topic": "standAlone_ACA",
            "dictionary": "present",
            "operational_table": "absent",
            "status": "dictionary-only field",
        },
        {
            "topic": "Spectral-resolution template",
            "dictionary": "Spec.Res SPW [N]",
            "operational_table": "Spec.Res. SPW N",
            "status": "explicit alias required",
        },
        {
            "topic": "Mosaic coordinate label",
            "dictionary": "Mos. Coord. Ref. Sys",
            "operational_table": "Mos. Coord.",
            "status": "explicit alias required",
        },
        {
            "topic": "Requested LAS label",
            "dictionary": "Req.LAS",
            "operational_table": "Req. LAS",
            "status": "explicit alias required",
        },
    ]
)

display(schema_discrepancies)

assert "standAlone_ACA" in set(queue_dictionary["Column Heading"])
assert "standAlone_ACA" not in set(queue_raw.columns)


assert "Req.LAS" in set(queue_dictionary["Column Heading"])
assert "Req. LAS" in set(queue_raw.columns)

,topic,dictionary,operational_table,status
0,SPS Bandwidth unit,[MHz],[GHz],conflict; preserve both
1,Velocity unit spelling,[km/s],[kms/s],lexical discrepancy
2,Mosaic datatype,[boolean],Custom / Rectangle / blank,"categorical, not boolean"
3,standAlone_ACA,present,absent,dictionary-only field
4,Spectral-resolution template,Spec.Res SPW [N],Spec.Res. SPW N,explicit alias required
5,Mosaic coordinate label,Mos. Coord. Ref. Sys,Mos. Coord.,explicit alias required
6,Requested LAS label,Req.LAS,Req. LAS,explicit alias required


## 5. Row granularity and identifiers

`Project Code`, target name, and band are useful grouping dimensions but are not
assumed to form a stable row key. Exact duplicate rows, repeated project-target-band
groups, coordinate offsets, and spectral setups are inspected before a queue candidate
identity is designed. Physical source line and content hash provide notebook-level
surrogate provenance.


In [13]:
exact_duplicate_mask = queue_raw.duplicated(keep=False)

project_target_band_groups = (
    queue_raw.groupby(
        ["Project Code", "Target Name", "Band"],
        dropna=False,
    )
    .size()
    .rename("row_count")
    .reset_index()
    .sort_values("row_count", ascending=False)
)

identity_summary = pd.Series(
    {
        "rows": len(queue_raw),
        "projects": queue_raw["Project Code"].nunique(),
        "project_target_pairs": queue_raw[
            ["Project Code", "Target Name"]
        ].drop_duplicates().shape[0],
        "project_target_band_groups": len(project_target_band_groups),
        "rows_participating_in_exact_duplicates": int(
            exact_duplicate_mask.sum()
        ),
        "unique_source_row_hashes": source_row_hashes.nunique(),
    }
)

display(identity_summary)
display(project_target_band_groups.head(20))
display(
    project_target_band_groups["row_count"]
    .value_counts()
    .sort_index()
    .rename_axis("rows_per_group")
    .reset_index(name="group_count")
)


rows                                      3200
projects                                    47
project_target_pairs                       403
project_target_band_groups                 419
rows_participating_in_exact_duplicates      75
unique_source_row_hashes                  3135
dtype: int64

,Project Code,Target Name,Band,row_count
290,2025.1.00539.S,M33,ALMA_RB_06,35
351,2025.1.00576.L,NGC_0253,ALMA_RB_06,23
60,2025.1.00383.L,AG301.1365-0.2259,ALMA_RB_03,14
57,2025.1.00383.L,AG300.3407-0.2197,ALMA_RB_03,14
59,2025.1.00383.L,AG300.7479+0.0976,ALMA_RB_03,14
58,2025.1.00383.L,AG300.4838-0.1042,ALMA_RB_03,14
62,2025.1.00383.L,AG301.6337-0.2355,ALMA_RB_03,14
61,2025.1.00383.L,AG301.2802-0.2244,ALMA_RB_03,14
63,2025.1.00383.L,AG301.6593-0.2494,ALMA_RB_03,14
56,2025.1.00383.L,AG300.1639-0.0889,ALMA_RB_03,14


,rows_per_group,group_count
0,1,181
1,2,3
2,4,1
3,7,42
4,11,1
5,14,189
6,23,1
7,35,1


In [14]:
if exact_duplicate_mask.any():
    exact_duplicate_examples = queue_raw.loc[
        exact_duplicate_mask,
        [
            "Project Code",
            "Target Name",
            "RA",
            "Dec",
            "Long Offset",
            "Lat Offset",
            "Mosaic",
            "Band",
            "Freq SPW 1",
        ],
    ].copy()
    exact_duplicate_examples["source_line_number"] = (
        source_line_numbers.loc[exact_duplicate_mask]
    )
    display(exact_duplicate_examples.head(30))
else:
    print("No exact duplicate source rows found.")

largest_group = project_target_band_groups.iloc[0]
largest_group_mask = (
    queue_raw["Project Code"].eq(largest_group["Project Code"])
    & queue_raw["Target Name"].eq(largest_group["Target Name"])
    & queue_raw["Band"].eq(largest_group["Band"])
)

display(
    queue_raw.loc[
        largest_group_mask,
        [
            "Project Code",
            "Target Name",
            "RA",
            "Dec",
            "Long Offset",
            "Lat Offset",
            "Mosaic",
            "Mos. Coord.",
            "Band",
            "Freq SPW 1",
            "Bandwidth SPW 1",
            "Spec.Res. SPW 1",
        ],
    ].head(40)
)


,Project Code,Target Name,RA,Dec,Long Offset,Lat Offset,Mosaic,Band,Freq SPW 1,source_line_number
3000,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3042
3001,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3043
3002,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3044
3003,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3045
3004,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3046
3005,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3047
3006,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3048
3007,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3049
3008,2025.1.00539.S,M33,23.536475,30.776986944444445,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3050
3009,2025.1.00539.S,M33,23.536475,30.776986944444445,0.0,0.0,Rectangle,ALMA_RB_06,232.0,3051


,Project Code,Target Name,RA,Dec,Long Offset,Lat Offset,Mosaic,Mos. Coord.,Band,Freq SPW 1,Bandwidth SPW 1,Spec.Res. SPW 1
3000,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25
3001,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25
3002,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25
3003,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25
3004,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25
3005,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25
3006,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25
3007,2025.1.00539.S,M33,23.482131,30.672143,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25
3008,2025.1.00539.S,M33,23.536475,30.776986944444445,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25
3009,2025.1.00539.S,M33,23.536475,30.776986944444445,0.0,0.0,Rectangle,ICRS,ALMA_RB_06,232.0,1875.0,31.25


## 6. Spatial–spectral factorization without a Cartesian assumption

Repeated queue rows contain several kinds of multiplicity: repeated pointings,
alternative spectral setups, request-level context, and exact exported duplicates.
The safe reconstruction is a factorized graph:

`RawQueueRow → (SpatialComponent, SpectralSetup, RequestContext)`

Signatures below are snapshot-local content identifiers. They support census and
deduplication of *entities*, but every relationship is still created from an observed
source row. A full Cartesian product may be reported as an empirical pattern; it is
never assumed by the parser.

In [15]:
SPATIAL_SIGNATURE_COLUMNS = [
    "RA",
    "Dec",
    "Long Offset",
    "Lat Offset",
    "Mosaic",
    "Mos. Length",
    "Mos. Width",
    "Mos. PA",
    "Mos. Spacing",
    "Mos. Coord.",
]

SPECTRAL_SIGNATURE_COLUMNS = [
    "Band",
    "Velocity",
    "Vel. Frame",
    "Vel. Convention",
    "Ref.Frequency",
    "Ref.Freq.Width",
    "Req.Sensitivity",
    "Is Sky Freq?",
    *sps_columns,
    *matched_spw_columns,
]

REQUEST_CONTEXT_COLUMNS = [
    "Req. Ang. Res.",
    "Req. LAS",
    "Use 7-m?",
    "Use TP?",
    "Polarization",
]

GROUP_COLUMNS = ["Project Code", "Target Name", "Band"]


def raw_signature(row, columns):
    payload = "\x1f".join(str(row[column]) for column in columns)
    return sha256(payload.encode("utf-8")).hexdigest()


factor_evidence = queue_raw[GROUP_COLUMNS].copy()
factor_evidence["source_line_number"] = source_line_numbers
factor_evidence["source_row_hash"] = source_row_hashes
factor_evidence["spatial_signature"] = queue_raw.apply(
    raw_signature,
    axis=1,
    columns=SPATIAL_SIGNATURE_COLUMNS,
)
factor_evidence["spectral_signature"] = queue_raw.apply(
    raw_signature,
    axis=1,
    columns=SPECTRAL_SIGNATURE_COLUMNS,
)
factor_evidence["context_signature"] = queue_raw.apply(
    raw_signature,
    axis=1,
    columns=REQUEST_CONTEXT_COLUMNS,
)

factor_group_records = []

for group_key, group in factor_evidence.groupby(
    GROUP_COLUMNS,
    dropna=False,
    sort=False,
):
    spatial_count = group["spatial_signature"].nunique()
    spectral_count = group["spectral_signature"].nunique()
    observed_pairs = group[
        ["spatial_signature", "spectral_signature"]
    ].drop_duplicates().shape[0]

    factor_group_records.append(
        {
            **dict(zip(GROUP_COLUMNS, group_key)),
            "raw_rows": len(group),
            "unique_raw_contents": group["source_row_hash"].nunique(),
            "spatial_signatures": spatial_count,
            "spectral_signatures": spectral_count,
            "context_signatures": group["context_signature"].nunique(),
            "observed_spatial_spectral_pairs": observed_pairs,
            "possible_cartesian_pairs": spatial_count * spectral_count,
            "exact_export_multiplicity": (
                len(group) - group["source_row_hash"].nunique()
            ),
        }
    )

factor_group_summary = pd.DataFrame(factor_group_records)
factor_group_summary["pair_coverage"] = (
    factor_group_summary["observed_spatial_spectral_pairs"]
    / factor_group_summary["possible_cartesian_pairs"]
)

factor_pattern_census = (
    factor_group_summary.groupby(
        [
            "raw_rows",
            "unique_raw_contents",
            "spatial_signatures",
            "spectral_signatures",
            "context_signatures",
            "observed_spatial_spectral_pairs",
            "possible_cartesian_pairs",
        ],
        dropna=False,
    )
    .size()
    .rename("group_count")
    .reset_index()
    .sort_values("group_count", ascending=False)
)

display(factor_pattern_census)

print("Groups:", len(factor_group_summary))
print(
    "Groups whose observed pairs fill the local product:",
    int(factor_group_summary["pair_coverage"].eq(1.0).sum()),
)
print(
    "Sparse/paired groups:",
    int(factor_group_summary["pair_coverage"].lt(1.0).sum()),
)
print(
    "Groups with repeated identical exports:",
    int(factor_group_summary["exact_export_multiplicity"].gt(0).sum()),
)

assert len(factor_group_summary) == 419
assert factor_group_summary["pair_coverage"].eq(1.0).sum() == 417
assert factor_group_summary["pair_coverage"].lt(1.0).sum() == 2
assert factor_group_summary["exact_export_multiplicity"].gt(0).sum() == 5

Groups: 419
Groups whose observed pairs fill the local product: 417
Sparse/paired groups: 2
Groups with repeated identical exports: 5


,raw_rows,unique_raw_contents,spatial_signatures,spectral_signatures,context_signatures,observed_spatial_spectral_pairs,possible_cartesian_pairs,group_count
6,14,14,7,2,2,14,14,189
0,1,1,1,1,1,1,1,181
4,7,7,7,1,1,7,7,42
2,2,2,1,2,1,2,2,2
1,2,1,1,1,1,1,1,1
3,4,1,1,1,1,1,1,1
5,11,1,1,1,1,1,1,1
7,23,2,2,2,1,2,4,1
8,35,5,5,5,5,5,25,1


In [16]:
sparse_factor_groups = factor_group_summary.loc[
    factor_group_summary["pair_coverage"].lt(1.0)
].sort_values("pair_coverage")

display(
    sparse_factor_groups[
        [
            *GROUP_COLUMNS,
            "raw_rows",
            "unique_raw_contents",
            "spatial_signatures",
            "spectral_signatures",
            "observed_spatial_spectral_pairs",
            "possible_cartesian_pairs",
            "pair_coverage",
            "exact_export_multiplicity",
        ]
    ]
)

association_multiplicity = (
    factor_evidence.groupby(
        [
            *GROUP_COLUMNS,
            "spatial_signature",
            "spectral_signature",
            "context_signature",
        ],
        dropna=False,
    )
    .agg(
        raw_row_multiplicity=("source_line_number", "size"),
        source_line_numbers=(
            "source_line_number",
            lambda values: tuple(int(value) for value in values),
        ),
        source_content_hashes=(
            "source_row_hash",
            lambda values: tuple(sorted(set(values))),
        ),
    )
    .reset_index()
)

display(
    association_multiplicity.loc[
        association_multiplicity["raw_row_multiplicity"].gt(1),
        [
            *GROUP_COLUMNS,
            "raw_row_multiplicity",
            "source_line_numbers",
        ],
    ].sort_values("raw_row_multiplicity", ascending=False).head(20)
)

assert association_multiplicity["raw_row_multiplicity"].sum() == len(queue_raw)
assert (
    association_multiplicity["raw_row_multiplicity"]
    - 1
).sum() == 65

,Project Code,Target Name,Band,raw_rows,unique_raw_contents,spatial_signatures,spectral_signatures,observed_spatial_spectral_pairs,possible_cartesian_pairs,pair_coverage,exact_export_multiplicity
290,2025.1.00539.S,M33,ALMA_RB_06,35,5,5,5,5,25,0.2,30
350,2025.1.00576.L,NGC_0253,ALMA_RB_06,23,2,2,2,2,4,0.5,21


,Project Code,Target Name,Band,raw_row_multiplicity,source_line_numbers
3066,2025.1.00576.L,NGC_0253,ALMA_RB_06,12,"(3136, 3137, 3138, 3139, 3140, 3141, 3142, 314..."
3065,2025.1.00576.L,NGC_0253,ALMA_RB_06,11,"(3166, 3167, 3168, 3169, 3170, 3171, 3172, 317..."
3068,2025.1.00576.L,NGC_5236,ALMA_RB_06,11,"(3152, 3153, 3154, 3155, 3156, 3157, 3158, 315..."
3000,2025.1.00539.S,M33,ALMA_RB_06,9,"(3058, 3059, 3060, 3061, 3062, 3063, 3064, 306..."
3004,2025.1.00539.S,M33,ALMA_RB_06,8,"(3069, 3070, 3071, 3072, 3073, 3074, 3075, 3076)"
3002,2025.1.00539.S,M33,ALMA_RB_06,8,"(3042, 3043, 3044, 3045, 3046, 3047, 3048, 3049)"
3003,2025.1.00539.S,M33,ALMA_RB_06,8,"(3050, 3051, 3052, 3053, 3054, 3055, 3056, 3057)"
3067,2025.1.00576.L,NGC_0300,ALMA_RB_06,4,"(3148, 3149, 3150, 3151)"
3001,2025.1.00539.S,M33,ALMA_RB_06,2,"(3067, 3068)"
3134,2025.1.01696.T,AT2026dbl,ALMA_RB_07,2,"(3240, 3241)"


### Factorization result

The common 14-row pattern is real: 189 groups contain seven spatial signatures
and two spectral signatures, and all 14 observed pairs are present. Another 42
groups contain seven spatial signatures with one spectral setup.

That does **not** establish a global Cartesian rule. Two groups are deliberately
sparse/paired:

- one has five spatial and five spectral signatures but only five observed pairs;
- one has two spatial and two spectral signatures but only two observed pairs.

Constructing the possible 25 or four pairs would invent observations that the file
does not contain. Exact duplicate exports are also retained through
`raw_row_multiplicity` and physical source lines rather than silently dropped.

## 7. Mosaic structure and tolerance closure

The dictionary labels `Mosaic` as boolean, but operational values are `Custom`,
`Rectangle`, and blank. Offset values also contain floating-point serialization
noise, so “non-zero” must be defined by a documented tolerance rather than exact
comparison.

The census below evaluates several candidate thresholds. The selected exploratory
zero tolerance is `1e-6 arcsec`: it is safely above the observed center noise
(below `1e-8 arcsec`) and far below the smallest clearly meaningful offset. This is
a CSV reconstruction tolerance, not the later half-power-beam duplication rule.

In [17]:
numeric_geometry_columns = [
    "Long Offset",
    "Lat Offset",
    "Mos. Length",
    "Mos. Width",
    "Mos. PA",
    "Mos. Spacing",
]

geometry_numeric = queue_raw[numeric_geometry_columns].apply(
    pd.to_numeric,
    errors="raise",
)

offset_radius_arcsec = np.hypot(
    geometry_numeric["Long Offset"],
    geometry_numeric["Lat Offset"],
).rename("offset_radius_arcsec")

MOSAIC_ZERO_TOLERANCE_ARCSEC = 1e-6
OFFSET_TOLERANCE_CANDIDATES_ARCSEC = [
    0.0,
    1e-12,
    1e-10,
    1e-9,
    1e-8,
    1e-6,
    1e-4,
    1e-2,
    1e-1,
    1.0,
]

mosaic_label = queue_raw["Mosaic"].replace({"": "<blank>"})
tolerance_records = []

for tolerance in OFFSET_TOLERANCE_CANDIDATES_ARCSEC:
    center_like = offset_radius_arcsec.le(tolerance)
    for label in ["Custom", "Rectangle", "<blank>"]:
        label_mask = mosaic_label.eq(label)
        tolerance_records.append(
            {
                "tolerance_arcsec": tolerance,
                "mosaic_label": label,
                "rows": int(label_mask.sum()),
                "center_like_rows": int((label_mask & center_like).sum()),
                "offset_rows": int((label_mask & ~center_like).sum()),
            }
        )

offset_tolerance_census = pd.DataFrame(tolerance_records)
display(
    offset_tolerance_census.pivot(
        index="tolerance_arcsec",
        columns="mosaic_label",
        values=["center_like_rows", "offset_rows"],
    )
)

has_nonzero_offset = offset_radius_arcsec.gt(
    MOSAIC_ZERO_TOLERANCE_ARCSEC
).rename("has_nonzero_offset")

has_rectangle_extent = (
    geometry_numeric["Mos. Length"].abs().gt(MOSAIC_ZERO_TOLERANCE_ARCSEC)
    | geometry_numeric["Mos. Width"].abs().gt(MOSAIC_ZERO_TOLERANCE_ARCSEC)
    | geometry_numeric["Mos. Spacing"].abs().gt(MOSAIC_ZERO_TOLERANCE_ARCSEC)
).rename("has_rectangle_extent")

selected_tolerance_summary = pd.crosstab(
    mosaic_label,
    has_nonzero_offset,
    margins=True,
)
display(selected_tolerance_summary)
display(pd.crosstab(mosaic_label, has_rectangle_extent, margins=True))
display(pd.crosstab(mosaic_label, queue_raw["Mos. Coord."], margins=True))

assert int((mosaic_label.eq("Custom") & ~has_nonzero_offset).sum()) == 420
assert int((mosaic_label.eq("Custom") & has_nonzero_offset).sum()) == 2520
assert int((mosaic_label.eq("Rectangle") & has_nonzero_offset).sum()) == 0
assert int((mosaic_label.eq("<blank>") & has_nonzero_offset).sum()) == 9
assert int((mosaic_label.eq("Rectangle") & has_rectangle_extent).sum()) == 140

center_like_rows                  offset_rows                 
mosaic_label              <blank> Custom Rectangle     <blank> Custom Rectangle
tolerance_arcsec                                                               
0.000000e+00                  109      0       140          11   2940         0
1.000000e-12                  109      0       140          11   2940         0
1.000000e-10                  111      0       140           9   2940         0
1.000000e-09                  111      0       140           9   2940         0
1.000000e-08                  111    417       140           9   2523         0
1.000000e-06                  111    420       140           9   2520         0
1.000000e-04                  111    420       140           9   2520         0
1.000000e-02                  111    420       140           9   2520         0
1.000000e-01                  112    420       140           8   2520         0
1.000000e+00                  115    420       140           5   2520         0

has_nonzero_offset,False,True,All
Mosaic,,,
<blank>,111,9,120
Custom,420,2520,2940
Rectangle,140,0,140
All,671,2529,3200


has_rectangle_extent,False,True,All
Mosaic,,,
<blank>,120,0,120
Custom,2940,0,2940
Rectangle,0,140,140
All,3060,140,3200


Mos. Coord.,,ICRS,galactic,All
Mosaic,,,,
<blank>,120,0,0,120
Custom,2940,0,0,2940
Rectangle,0,138,2,140
All,3060,138,2,3200


In [18]:
mosaic_diagnostic = queue_raw.loc[
    :,
    [
        "Project Code",
        "Target Name",
        "RA",
        "Dec",
        "Long Offset",
        "Lat Offset",
        "Mosaic",
        "Mos. Length",
        "Mos. Width",
        "Mos. PA",
        "Mos. Spacing",
        "Mos. Coord.",
        "Band",
    ],
].copy()
mosaic_diagnostic["offset_radius_arcsec"] = offset_radius_arcsec
mosaic_diagnostic["has_nonzero_offset"] = has_nonzero_offset
mosaic_diagnostic["has_rectangle_extent"] = has_rectangle_extent
mosaic_diagnostic["spatial_signature"] = factor_evidence["spatial_signature"]
mosaic_diagnostic["source_line_number"] = source_line_numbers

custom_unique_spatial = (
    mosaic_diagnostic.loc[mosaic_diagnostic["Mosaic"].eq("Custom")]
    .drop_duplicates(
        ["Project Code", "Target Name", "Band", "spatial_signature"]
    )
    .assign(
        center_like=lambda frame: frame["offset_radius_arcsec"].le(
            MOSAIC_ZERO_TOLERANCE_ARCSEC
        )
    )
)

custom_group_geometry = (
    custom_unique_spatial.groupby(GROUP_COLUMNS, dropna=False)
    .agg(
        unique_spatial_points=("spatial_signature", "size"),
        center_like_points=("center_like", "sum"),
        minimum_radius_arcsec=("offset_radius_arcsec", "min"),
        maximum_radius_arcsec=("offset_radius_arcsec", "max"),
    )
    .reset_index()
)
custom_group_geometry["offset_points"] = (
    custom_group_geometry["unique_spatial_points"]
    - custom_group_geometry["center_like_points"]
)

display(
    custom_group_geometry[
        [
            "unique_spatial_points",
            "center_like_points",
            "offset_points",
        ]
    ].value_counts().rename("group_count").reset_index()
)
display(
    offset_radius_arcsec.groupby(mosaic_label).quantile(
        [0.0, 0.01, 0.5, 0.99, 1.0]
    ).rename("offset_radius_arcsec")
)

assert len(custom_group_geometry) == 231
assert custom_group_geometry["unique_spatial_points"].eq(7).all()
assert custom_group_geometry["center_like_points"].eq(1).all()
assert custom_group_geometry["offset_points"].eq(6).all()

,unique_spatial_points,center_like_points,offset_points,group_count
0,7,1,6,231


Mosaic         
<blank>    0.00    0.000000e+00
           0.01    0.000000e+00
           0.50    0.000000e+00
           0.99    8.323358e+00
           1.00    2.252780e+01
Custom     0.00    1.128485e-09
           0.01    3.392606e-09
           0.50    3.192998e+01
           0.99    3.192998e+01
           1.00    3.192998e+01
Rectangle  0.00    0.000000e+00
           0.01    0.000000e+00
           0.50    0.000000e+00
           0.99    0.000000e+00
           1.00    0.000000e+00
Name: offset_radius_arcsec, dtype: float64

### Mosaic result

At `1e-6 arcsec`, every one of the 231 custom-mosaic groups has exactly one
center-like spatial component and six genuine offset components. The 2,940 raw
custom rows contain 420 center copies and 2,520 offset copies because spectral
setups and export multiplicity repeat the same spatial entities.

All 140 rectangle rows have rectangle extents and center offsets. Blank `Mosaic`
usually represents a single pointing, but nine blank rows carry meaningful offsets;
therefore blank is not equivalent to “all spatial values are zero.” The raw values
and the tolerance used to classify them must both remain available.

## 8. Frequency reference and velocity context

`Is Sky Freq?` determines whether SPW/SPS frequencies are already sky frequencies or
are rest frequencies. Rest-frequency rows require a convention-specific Doppler
derivation using velocity, velocity frame, and convention. The diagnostic derivation
below preserves both input and output values.

The derived frequency is expressed according to the row's declared velocity frame; it
is not automatically equivalent to the frequency frame used by every Archive record.
Frame alignment and the final comparison policy therefore remain separate work.


In [19]:
velocity_kms = pd.to_numeric(queue_raw["Velocity"], errors="coerce")

velocity_context_census = pd.crosstab(
    [queue_raw["Is Sky Freq?"], queue_raw["Vel. Frame"]],
    queue_raw["Vel. Convention"],
    margins=True,
)

frequency_kind_census = (
    queue_raw["Is Sky Freq?"]
    .value_counts()
    .rename_axis("is_sky_frequency_raw")
    .reset_index(name="row_count")
)
frequency_kind_census["fraction"] = (
    frequency_kind_census["row_count"] / len(queue_raw)
)

display(frequency_kind_census)
display(velocity_context_census)

print("Unparseable velocities:", int(velocity_kms.isna().sum()))
print("Minimum velocity [km/s]:", velocity_kms.min())
print("Maximum velocity [km/s]:", velocity_kms.max())

assert velocity_kms.notna().all()

velocity_domain_by_convention = (
    pd.DataFrame(
        {
            "convention": queue_raw["Vel. Convention"],
            "velocity_kms": velocity_kms,
        }
    )
    .groupby("convention")["velocity_kms"]
    .agg(["count", "min", "max"])
)
display(velocity_domain_by_convention)

# These are convention coordinates, not one uniform physical-speed quantity.
# Optical cz values may exceed c. Validate the mathematical domain separately.
radio_velocity = velocity_kms[queue_raw["Vel. Convention"].eq("RADIO")]
optical_velocity = velocity_kms[queue_raw["Vel. Convention"].eq("OPTICAL")]
relativistic_velocity = velocity_kms[
    queue_raw["Vel. Convention"].eq("RELATIVISTIC")
]

assert radio_velocity.lt(299792.458).all()
assert optical_velocity.gt(-299792.458).all()
assert relativistic_velocity.abs().lt(299792.458).all()


Unparseable velocities: 0
Minimum velocity [km/s]: -180.0
Maximum velocity [km/s]: 1384621.4465188


,is_sky_frequency_raw,row_count,fraction
0,False,3164,0.98875
1,True,36,0.01125


Vel. Convention          OPTICAL  RADIO  RELATIVISTIC   All
Is Sky Freq? Vel. Frame                                    
False        hel               0      0           108   108
             lsrk              5   3045             1  3051
             topo              0      5             0     5
True         hel               0      0             1     1
             lsrk              8     26             0    34
             topo              0      1             0     1
All                           13   3077           110  3200

,count,min,max
convention,,,
OPTICAL,13,7338.919372,1.384621e+06
RADIO,3077,-90.100000,2.748368e+05
RELATIVISTIC,110,-180.000000,2.739169e+05


In [20]:
C_KMS = 299792.458


def rest_to_observed_frequency_ghz(
    rest_frequency_ghz,
    velocity_kms,
    convention,
):
    # Diagnostic Doppler conversion in the row's declared velocity frame.
    rest = np.asarray(rest_frequency_ghz, dtype=float)
    velocity = np.asarray(velocity_kms, dtype=float)
    convention_values = np.asarray(convention, dtype=str)
    beta = velocity / C_KMS

    observed = np.full(rest.shape, np.nan, dtype=float)

    radio_mask = convention_values == "RADIO"
    optical_mask = convention_values == "OPTICAL"
    relativistic_mask = convention_values == "RELATIVISTIC"

    if np.any(beta[radio_mask] >= 1):
        raise ValueError("RADIO convention requires v/c < 1.")
    if np.any(1.0 + beta[optical_mask] <= 0):
        raise ValueError("OPTICAL convention requires 1 + v/c > 0.")
    if np.any(np.abs(beta[relativistic_mask]) >= 1):
        raise ValueError("RELATIVISTIC convention requires |v/c| < 1.")

    observed[radio_mask] = rest[radio_mask] * (1.0 - beta[radio_mask])
    observed[optical_mask] = rest[optical_mask] / (1.0 + beta[optical_mask])
    observed[relativistic_mask] = rest[relativistic_mask] * np.sqrt(
        (1.0 - beta[relativistic_mask])
        / (1.0 + beta[relativistic_mask])
    )

    supported = radio_mask | optical_mask | relativistic_mask
    if not np.all(supported):
        unknown = sorted(set(convention_values[~supported]))
        raise ValueError(f"Unsupported velocity conventions: {unknown}")

    return observed


spw_frequency_diagnostic = queue_spw_long.copy()
spw_frequency_diagnostic["velocity_kms"] = pd.to_numeric(
    spw_frequency_diagnostic["velocity_raw"],
    errors="raise",
)
spw_frequency_diagnostic["is_sky_frequency"] = (
    spw_frequency_diagnostic["is_sky_frequency_raw"]
    .map({"True": True, "False": False})
)

assert spw_frequency_diagnostic["is_sky_frequency"].notna().all()

rest_mask = ~spw_frequency_diagnostic["is_sky_frequency"]
sky_mask = spw_frequency_diagnostic["is_sky_frequency"]

spw_frequency_diagnostic["derived_sky_frequency_ghz"] = (
    spw_frequency_diagnostic["frequency_ghz"]
)

spw_frequency_diagnostic.loc[
    rest_mask,
    "derived_sky_frequency_ghz",
] = rest_to_observed_frequency_ghz(
    spw_frequency_diagnostic.loc[rest_mask, "frequency_ghz"],
    spw_frequency_diagnostic.loc[rest_mask, "velocity_kms"],
    spw_frequency_diagnostic.loc[
        rest_mask,
        "velocity_convention_raw",
    ],
)

spw_frequency_diagnostic["frequency_derivation"] = np.where(
    sky_mask,
    "identity_from_declared_sky_frequency",
    "derived_from_rest_frequency_and_velocity_convention",
)

assert spw_frequency_diagnostic["derived_sky_frequency_ghz"].notna().all()
assert spw_frequency_diagnostic["derived_sky_frequency_ghz"].gt(0).all()

display(
    spw_frequency_diagnostic[
        [
            "project_code",
            "target_name",
            "band",
            "spw_number",
            "frequency_raw",
            "is_sky_frequency_raw",
            "velocity_raw",
            "velocity_frame_raw",
            "velocity_convention_raw",
            "derived_sky_frequency_ghz",
            "frequency_derivation",
        ]
    ].head(20)
)


,project_code,target_name,band,spw_number,frequency_raw,is_sky_frequency_raw,velocity_raw,velocity_frame_raw,velocity_convention_raw,derived_sky_frequency_ghz,frequency_derivation
0,2024.1.00750.S,NGC6240,ALMA_RB_07,1,350.4999999999,True,7338.919371839999,lsrk,OPTICAL,350.500000,identity_from_declared_sky_frequency
1,2024.1.00750.S,NGC6240,ALMA_RB_07,2,348.5000000001,True,7338.919371839999,lsrk,OPTICAL,348.500000,identity_from_declared_sky_frequency
2,2024.1.00750.S,NGC6240,ALMA_RB_07,3,338.4999999999,True,7338.919371839999,lsrk,OPTICAL,338.500000,identity_from_declared_sky_frequency
3,2024.1.00750.S,NGC6240,ALMA_RB_07,4,336.5000000001,True,7338.919371839999,lsrk,OPTICAL,336.500000,identity_from_declared_sky_frequency
4,2024.1.01166.S,"Tsuchinshan-ATLAS_C2023_A3 Epoch 2, band 1",ALMA_RB_01,1,43.188,True,0.0,topo,RADIO,43.188000,identity_from_declared_sky_frequency
5,2024.1.01166.S,"Tsuchinshan-ATLAS_C2023_A3 Epoch 2, band 1",ALMA_RB_01,2,41.188,True,0.0,topo,RADIO,41.188000,identity_from_declared_sky_frequency
6,2024.1.01166.S,"Tsuchinshan-ATLAS_C2023_A3 Epoch 2, band 1",ALMA_RB_01,3,39.125,True,0.0,topo,RADIO,39.125000,identity_from_declared_sky_frequency
7,2024.1.01166.S,"Tsuchinshan-ATLAS_C2023_A3 Epoch 2, band 1",ALMA_RB_01,4,37.188,True,0.0,topo,RADIO,37.188000,identity_from_declared_sky_frequency
8,2024.1.01166.S,"Tsuchinshan-ATLAS_C2023_A3 Copy of Epoch 2, ba...",ALMA_RB_07,1,345.79599,False,0.0,topo,RADIO,345.795990,derived_from_rest_frequency_and_velocity_conve...
9,2024.1.01166.S,"Tsuchinshan-ATLAS_C2023_A3 Copy of Epoch 2, ba...",ALMA_RB_07,2,354.505473,False,0.0,topo,RADIO,354.505473,derived_from_rest_frequency_and_velocity_conve...


In [21]:
def observed_to_velocity_kms(
    rest_frequency_ghz,
    observed_frequency_ghz,
    convention,
):
    # Inverse diagnostic used only to validate the formulas above.
    rest = np.asarray(rest_frequency_ghz, dtype=float)
    observed = np.asarray(observed_frequency_ghz, dtype=float)
    convention_values = np.asarray(convention, dtype=str)
    beta = np.full(rest.shape, np.nan, dtype=float)

    radio_mask = convention_values == "RADIO"
    optical_mask = convention_values == "OPTICAL"
    relativistic_mask = convention_values == "RELATIVISTIC"

    beta[radio_mask] = 1.0 - observed[radio_mask] / rest[radio_mask]
    beta[optical_mask] = rest[optical_mask] / observed[optical_mask] - 1.0
    beta[relativistic_mask] = (
        rest[relativistic_mask] ** 2
        - observed[relativistic_mask] ** 2
    ) / (
        rest[relativistic_mask] ** 2
        + observed[relativistic_mask] ** 2
    )

    return beta * C_KMS


round_trip_velocity = observed_to_velocity_kms(
    spw_frequency_diagnostic.loc[rest_mask, "frequency_ghz"],
    spw_frequency_diagnostic.loc[
        rest_mask,
        "derived_sky_frequency_ghz",
    ],
    spw_frequency_diagnostic.loc[
        rest_mask,
        "velocity_convention_raw",
    ],
)

round_trip_error_kms = np.abs(
    round_trip_velocity
    - spw_frequency_diagnostic.loc[rest_mask, "velocity_kms"].to_numpy()
)

print("Rest-frequency SPW evidence rows:", int(rest_mask.sum()))
print("Declared-sky SPW evidence rows:", int(sky_mask.sum()))
print("Maximum Doppler round-trip error [km/s]:", round_trip_error_kms.max())

assert round_trip_error_kms.max() < 1e-6

high_redshift_example = spw_frequency_diagnostic.loc[
    spw_frequency_diagnostic["target_name"].eq("FS_z8.3"),
    [
        "project_code",
        "target_name",
        "band",
        "spw_number",
        "frequency_ghz",
        "velocity_kms",
        "velocity_frame_raw",
        "velocity_convention_raw",
        "derived_sky_frequency_ghz",
    ],
]

display(high_redshift_example)


Rest-frequency SPW evidence rows: 16076
Declared-sky SPW evidence rows: 140
Maximum Doppler round-trip error [km/s]: 2.3283064365386963e-10


,project_code,target_name,band,spw_number,frequency_ghz,velocity_kms,velocity_frame_raw,velocity_convention_raw,derived_sky_frequency_ghz
22,2024.1.01483.S,FS_z8.3,ALMA_RB_07,1,3272.073323,267570.568705,lsrk,RADIO,351.684579
23,2024.1.01483.S,FS_z8.3,ALMA_RB_07,2,3288.819789,267570.568705,lsrk,RADIO,353.484500
24,2024.1.01483.S,FS_z8.3,ALMA_RB_07,3,3382.786068,267570.568705,lsrk,RADIO,363.584057
25,2024.1.01483.S,FS_z8.3,ALMA_RB_07,4,3399.067354,267570.568705,lsrk,RADIO,365.333980


### Reference-frequency interval validation

For each regular row, every raw SPW center is first converted to the row's declared
sky-frequency representation when required. `Bandwidth SPW N` remains the nominal
correlator width and is converted only from MHz to GHz; it is not multiplied by the
center-frequency Doppler factor. `Ref.Frequency` must lie inside at least one
resulting interval.

This is a strong internal CSV consistency check: it independently connects the
sensitivity reference frequency to the derived SPWs. It still does not prove that an
Archive observation and the queue row use an identical reference frame.

In [22]:
spw_interval_diagnostic = spw_frequency_diagnostic.copy()
spw_interval_diagnostic["reference_frequency_ghz"] = (
    pd.to_numeric(queue_raw["Ref.Frequency"], errors="raise")
    .reindex(spw_interval_diagnostic["source_row_index"])
    .to_numpy()
)

# Bandwidth SPW N is the nominal correlator width. Only the centre
# frequency is Doppler converted for rest-frequency rows.
spw_interval_diagnostic["nominal_bandwidth_ghz"] = (
    spw_interval_diagnostic["bandwidth_mhz"]
    / 1000.0
)
spw_interval_diagnostic["lower_sky_frequency_ghz"] = (
    spw_interval_diagnostic["derived_sky_frequency_ghz"]
    - spw_interval_diagnostic["nominal_bandwidth_ghz"] / 2.0
)
spw_interval_diagnostic["upper_sky_frequency_ghz"] = (
    spw_interval_diagnostic["derived_sky_frequency_ghz"]
    + spw_interval_diagnostic["nominal_bandwidth_ghz"] / 2.0
)

# Reproduced from getUsableBandwidth() in the portal-provided Cycle 13
# plotobs_cycle13.py v1.3.1 script. Its processor scope remains pending.
USABLE_BANDWIDTH_DERIVATION_VERSION = (
    "cycle13-portal-plotobs-v1.3.1-v1"
)
USABLE_BANDWIDTH_MHZ = {
    62.5: 58.6,
    125.0: 117.2,
    250.0: 234.4,
    500.0: 468.8,
    1000.0: 937.5,
    1875.0: 1875.0,
    2000.0: 1875.0,
}
spw_interval_diagnostic["usable_bandwidth_mhz"] = (
    spw_interval_diagnostic["bandwidth_mhz"].map(
        USABLE_BANDWIDTH_MHZ
    )
)
assert spw_interval_diagnostic["usable_bandwidth_mhz"].notna().all()
spw_interval_diagnostic["usable_bandwidth_ghz"] = (
    spw_interval_diagnostic["usable_bandwidth_mhz"] / 1000.0
)
spw_interval_diagnostic["usable_lower_sky_frequency_ghz"] = (
    spw_interval_diagnostic["derived_sky_frequency_ghz"]
    - spw_interval_diagnostic["usable_bandwidth_ghz"] / 2.0
)
spw_interval_diagnostic["usable_upper_sky_frequency_ghz"] = (
    spw_interval_diagnostic["derived_sky_frequency_ghz"]
    + spw_interval_diagnostic["usable_bandwidth_ghz"] / 2.0
)
spw_interval_diagnostic["contains_reference_frequency"] = (
    spw_interval_diagnostic["reference_frequency_ghz"].ge(
        spw_interval_diagnostic["lower_sky_frequency_ghz"] - 1e-12
    )
    & spw_interval_diagnostic["reference_frequency_ghz"].le(
        spw_interval_diagnostic["upper_sky_frequency_ghz"] + 1e-12
    )
)
spw_interval_diagnostic["contains_reference_frequency_usable"] = (
    spw_interval_diagnostic["reference_frequency_ghz"].ge(
        spw_interval_diagnostic["usable_lower_sky_frequency_ghz"] - 1e-12
    )
    & spw_interval_diagnostic["reference_frequency_ghz"].le(
        spw_interval_diagnostic["usable_upper_sky_frequency_ghz"] + 1e-12
    )
)
spw_interval_diagnostic["center_difference_ghz"] = (
    spw_interval_diagnostic["reference_frequency_ghz"]
    - spw_interval_diagnostic["derived_sky_frequency_ghz"]
).abs()

row_interval_validation = (
    spw_interval_diagnostic.groupby("source_row_index")
    .agg(
        reference_frequency_ghz=("reference_frequency_ghz", "first"),
        containing_spw_count=("contains_reference_frequency", "sum"),
        usable_containing_spw_count=(
            "contains_reference_frequency_usable",
            "sum",
        ),
        minimum_center_difference_ghz=("center_difference_ghz", "min"),
    )
    .join(source_line_numbers)
)
row_interval_validation["reference_inside_any_spw"] = (
    row_interval_validation["containing_spw_count"].gt(0)
)
row_interval_validation["reference_inside_any_usable_spw"] = (
    row_interval_validation["usable_containing_spw_count"].gt(0)
)

center_difference_thresholds_ghz = [1e-9, 1e-6, 1e-4, 1e-1, 1.0]
center_difference_census = pd.DataFrame(
    {
        "threshold_ghz": center_difference_thresholds_ghz,
        "rows_at_or_below_threshold": [
            int(
                row_interval_validation[
                    "minimum_center_difference_ghz"
                ].le(threshold).sum()
            )
            for threshold in center_difference_thresholds_ghz
        ],
    }
)
center_difference_census["fraction_of_regular_rows"] = (
    center_difference_census["rows_at_or_below_threshold"]
    / len(row_interval_validation)
)

display(center_difference_census)
display(
    row_interval_validation.sort_values(
        "minimum_center_difference_ghz",
        ascending=False,
    ).head(10)
)

print("Regular rows validated:", len(row_interval_validation))
print(
    "Reference frequency inside at least one derived interval:",
    int(row_interval_validation["reference_inside_any_spw"].sum()),
)
print(
    "Reference frequency inside at least one usable interval:",
    int(
        row_interval_validation[
            "reference_inside_any_usable_spw"
        ].sum()
    ),
)

assert len(row_interval_validation) == 3199
assert row_interval_validation["reference_inside_any_spw"].all()
assert row_interval_validation["reference_inside_any_usable_spw"].all()
assert center_difference_census["rows_at_or_below_threshold"].tolist() == [
    3192,
    3194,
    3197,
    3198,
    3199,
]

capers_spws = spw_interval_diagnostic.loc[
    spw_interval_diagnostic["project_code"].eq("2025.1.00806.S")
    & spw_interval_diagnostic["target_name"].eq("CAPERS_UDS_z11")
]
assert len(capers_spws) == 4
assert np.allclose(capers_spws["nominal_bandwidth_ghz"], 1.875)
assert np.allclose(capers_spws["usable_bandwidth_ghz"], 1.875)
assert np.allclose(
    capers_spws["spectral_resolution_mhz"],
    7.81201171875,
)
print("CAPERS nominal bandwidths [GHz]:", capers_spws["nominal_bandwidth_ghz"].tolist())
print("CAPERS usable bandwidths [GHz]:", capers_spws["usable_bandwidth_ghz"].tolist())
print("CAPERS spectral resolutions [MHz]:", capers_spws["spectral_resolution_mhz"].tolist())

Regular rows validated: 3199
Reference frequency inside at least one derived interval: 3199
Reference frequency inside at least one usable interval: 3199
CAPERS nominal bandwidths [GHz]: [1.875, 1.875, 1.875, 1.875]
CAPERS usable bandwidths [GHz]: [1.875, 1.875, 1.875, 1.875]
CAPERS spectral resolutions [MHz]: [7.81201171875, 7.81201171875, 7.81201171875, 7.81201171875]


,threshold_ghz,rows_at_or_below_threshold,fraction_of_regular_rows
0,1.000000e-09,3192,0.997812
1,1.000000e-06,3194,0.998437
2,1.000000e-04,3197,0.999375
3,1.000000e-01,3198,0.999687
4,1.000000e+00,3199,1.000000


,reference_frequency_ghz,containing_spw_count,usable_containing_spw_count,minimum_center_difference_ghz,source_line_number,reference_inside_any_spw,reference_inside_any_usable_spw
source_row_index,,,,,,,
3189,349.700000,1,1,9.295100e-01,3231,True,True
3192,302.190000,1,1,1.789000e-02,3234,True,True
3135,329.317807,1,1,2.999884e-06,3177,True,True
55,183.036496,1,1,2.995522e-06,97,True,True
3153,356.712451,1,1,1.999878e-06,3195,True,True
3168,282.812820,1,1,7.116248e-07,3210,True,True
2997,292.185918,1,1,5.014472e-07,3039,True,True
3164,663.800000,1,1,1.000444e-10,3206,True,True
3165,660.900000,1,1,1.000444e-10,3207,True,True


### Frequency-conversion boundary

The round-trip test checks that each convention-specific formula is internally
invertible for this snapshot. It does not establish that the derived value is already
in the exact frame required for comparison with Archive metadata. Production use
requires documented frame semantics, reference cases, and agreement with the
supervisor. Raw rest frequency and every derivation input must remain available.

`Velocity` must not be validated as one ordinary physical-speed column. In particular,
the optical convention commonly stores the coordinate $cz$, which may numerically
exceed $c$ without describing superluminal motion. Mathematical-domain checks are
therefore convention-specific.


## 9. Requested-sensitivity evidence and SPS basis

The embedded dictionary defines one proposal-side sensitivity triple:

- `Ref.Frequency [GHz]`: reference frequency for the requested sensitivity;
- `Ref.Freq.Width [MHz]`: bandwidth over which that sensitivity is requested;
- `Req.Sensitivity [mJy]`: user-requested RMS sensitivity.

This interpretation is consistent with the Cycle 13 Proposer's Guide and ALMA OT
documentation: the bandwidth used for sensitivity can be an aggregate bandwidth, a
representative-window resolution, or a user-defined smoothing width. It is not
necessarily equal to either native spectral resolution or SPW bandwidth.

Official references:

- https://almascience.eso.org/documents-and-tools/cycle13/alma-proposers-guide
- https://almascience.eso.org/documents-and-tools/cycle13/alma-ot-refmanual

The following census tests completeness, positivity, group variation, and only the
relationships that can be demonstrated from this snapshot. It does not infer
correlator mode from the requested width.

In [23]:
SENSITIVITY_COLUMNS = [
    "Ref.Frequency",
    "Ref.Freq.Width",
    "Req.Sensitivity",
]

sensitivity_numeric = queue_raw[SENSITIVITY_COLUMNS].apply(
    pd.to_numeric,
    errors="raise",
)

sensitivity_completeness = pd.DataFrame(
    {
        "column": SENSITIVITY_COLUMNS,
        "nonempty_rows": [
            int(queue_raw[column].astype(str).str.strip().ne("").sum())
            for column in SENSITIVITY_COLUMNS
        ],
        "positive_rows": [
            int(sensitivity_numeric[column].gt(0).sum())
            for column in SENSITIVITY_COLUMNS
        ],
        "distinct_values": [
            int(sensitivity_numeric[column].nunique())
            for column in SENSITIVITY_COLUMNS
        ],
        "minimum": [
            float(sensitivity_numeric[column].min())
            for column in SENSITIVITY_COLUMNS
        ],
        "maximum": [
            float(sensitivity_numeric[column].max())
            for column in SENSITIVITY_COLUMNS
        ],
    }
)
display(sensitivity_completeness)

sensitivity_group_variation = []
for column in SENSITIVITY_COLUMNS:
    distinct_per_group = queue_raw.groupby(
        GROUP_COLUMNS,
        dropna=False,
    )[column].nunique(dropna=False)
    sensitivity_group_variation.append(
        {
            "column": column,
            "groups": len(distinct_per_group),
            "groups_with_multiple_values": int(distinct_per_group.gt(1).sum()),
            "maximum_values_in_one_group": int(distinct_per_group.max()),
        }
    )

sensitivity_group_variation = pd.DataFrame(sensitivity_group_variation)
display(sensitivity_group_variation)

assert sensitivity_completeness["nonempty_rows"].eq(3200).all()
assert sensitivity_completeness["positive_rows"].eq(3200).all()
assert sensitivity_group_variation.set_index("column").loc[
    "Ref.Frequency", "groups_with_multiple_values"
] == 192
assert sensitivity_group_variation.set_index("column").loc[
    "Ref.Freq.Width", "groups_with_multiple_values"
] == 191
assert sensitivity_group_variation.set_index("column").loc[
    "Req.Sensitivity", "groups_with_multiple_values"
] == 191

,column,nonempty_rows,positive_rows,distinct_values,minimum,maximum
0,Ref.Frequency,3200,3200,443,43.188000,870.910168
1,Ref.Freq.Width,3200,3200,83,0.057882,15000.000000
2,Req.Sensitivity,3200,3200,101,0.004000,135.616155


,column,groups,groups_with_multiple_values,maximum_values_in_one_group
0,Ref.Frequency,419,192,2
1,Ref.Freq.Width,419,191,2
2,Req.Sensitivity,419,191,5


In [24]:
def union_interval_width_mhz(intervals_ghz):
    intervals = sorted(intervals_ghz)
    start, end = intervals[0]
    total_ghz = 0.0

    for next_start, next_end in intervals[1:]:
        if next_start <= end:
            end = max(end, next_end)
        else:
            total_ghz += end - start
            start, end = next_start, next_end

    total_ghz += end - start
    return total_ghz * 1000.0


regular_sensitivity_records = []

for row_index, group in spw_interval_diagnostic.groupby(
    "source_row_index",
    sort=False,
):
    reference_width_mhz = float(queue_raw.at[row_index, "Ref.Freq.Width"])
    raw_spectral_resolutions_mhz = group[
        "spectral_resolution_mhz"
    ].to_numpy()
    nominal_aggregate_nonoverlap_bandwidth_mhz = union_interval_width_mhz(
        list(
            zip(
                group["lower_sky_frequency_ghz"],
                group["upper_sky_frequency_ghz"],
            )
        )
    )

    usable_aggregate_nonoverlap_bandwidth_mhz = union_interval_width_mhz(
        list(
            zip(
                group["usable_lower_sky_frequency_ghz"],
                group["usable_upper_sky_frequency_ghz"],
            )
        )
    )

    matches_usable_aggregate = np.isclose(
        reference_width_mhz,
        usable_aggregate_nonoverlap_bandwidth_mhz,
        rtol=1e-7,
        atol=1e-6,
    )
    matches_raw_spw_resolution = np.isclose(
        reference_width_mhz,
        raw_spectral_resolutions_mhz,
        rtol=1e-7,
        atol=1e-6,
    ).any()

    if matches_usable_aggregate:
        observed_relation = "matches_usable_aggregate_bandwidth"
    elif matches_raw_spw_resolution:
        observed_relation = "matches_a_raw_spw_spectral_resolution"
    else:
        observed_relation = "other_or_user_defined_reference_width"

    regular_sensitivity_records.append(
        {
            "source_row_index": int(row_index),
            "source_line_number": int(source_line_numbers[row_index]),
            "project_code": queue_raw.at[row_index, "Project Code"],
            "target_name": queue_raw.at[row_index, "Target Name"],
            "reference_frequency_ghz": float(
                queue_raw.at[row_index, "Ref.Frequency"]
            ),
            "reference_width_mhz": reference_width_mhz,
            "requested_sensitivity_mjy": float(
                queue_raw.at[row_index, "Req.Sensitivity"]
            ),
            "nominal_aggregate_nonoverlap_bandwidth_mhz": (
                nominal_aggregate_nonoverlap_bandwidth_mhz
            ),
            "usable_aggregate_nonoverlap_bandwidth_mhz": (
                usable_aggregate_nonoverlap_bandwidth_mhz
            ),
            "observed_reference_width_relation": observed_relation,
        }
    )

regular_sensitivity_diagnostic = pd.DataFrame(
    regular_sensitivity_records
)

reference_width_relation_census = (
    regular_sensitivity_diagnostic[
        "observed_reference_width_relation"
    ]
    .value_counts()
    .rename_axis("observed_relation")
    .reset_index(name="row_count")
)
display(reference_width_relation_census)
display(
    regular_sensitivity_diagnostic.loc[
        regular_sensitivity_diagnostic[
            "observed_reference_width_relation"
        ].ne("other_or_user_defined_reference_width")
    ].head(40)
)

relation_counts = reference_width_relation_census.set_index(
    "observed_relation"
)["row_count"]

assert len(regular_sensitivity_diagnostic) == 3199
assert relation_counts["matches_usable_aggregate_bandwidth"] == 38
assert relation_counts["matches_a_raw_spw_spectral_resolution"] == 3
assert relation_counts["other_or_user_defined_reference_width"] == 3158

,observed_relation,row_count
0,other_or_user_defined_reference_width,3158
1,matches_usable_aggregate_bandwidth,38
2,matches_a_raw_spw_spectral_resolution,3


,source_row_index,source_line_number,project_code,target_name,reference_frequency_ghz,reference_width_mhz,requested_sensitivity_mjy,nominal_aggregate_nonoverlap_bandwidth_mhz,usable_aggregate_nonoverlap_bandwidth_mhz,observed_reference_width_relation
0,0,42,2024.1.00750.S,NGC6240,338.500000,7500.000000,0.008000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth
8,8,50,2024.1.01768.T,ToO1,104.500000,7500.000000,0.015000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth
10,10,52,2024.1.01780.T,GRB1,92.500000,7500.000000,0.025000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth
13,13,55,2024.A.00044.T,V462_Lup,350.500000,7500.000000,1.000000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth
14,14,56,2024.A.00044.T,V572_Vel,350.500000,7500.000000,1.000000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth
15,15,57,2024.A.00044.T,V462_Lup,152.000000,7500.000000,1.000000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth
16,16,58,2024.A.00044.T,V572_Vel,152.000000,7500.000000,1.000000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth
17,17,59,2024.A.00044.T,V462_Lup,43.188000,7500.000000,1.000000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth
18,18,60,2024.A.00044.T,V572_Vel,43.188000,7500.000000,1.000000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth
19,19,61,2024.A.00064.S,TYC_5709-354-1,348.500000,7500.000000,0.033000,7500.000000,7500.000000,matches_usable_aggregate_bandwidth


In [25]:
sps_sensitivity = queue_raw.loc[
    spectral_scan_mask,
    [
        "Project Code",
        "Target Name",
        "Ref.Frequency",
        "Ref.Freq.Width",
        "Req.Sensitivity",
        *sps_columns,
    ],
].copy()

for column in [
    "Ref.Frequency",
    "Ref.Freq.Width",
    "Req.Sensitivity",
    *sps_columns,
]:
    sps_sensitivity[column] = pd.to_numeric(
        sps_sensitivity[column],
        errors="raise",
    )

sps_sensitivity["reference_inside_scan_range"] = (
    sps_sensitivity["Ref.Frequency"].ge(
        sps_sensitivity["SPS Start Freq."]
    )
    & sps_sensitivity["Ref.Frequency"].le(
        sps_sensitivity["SPS End Freq."]
    )
)
sps_sensitivity["reference_width_to_native_resolution_ratio"] = (
    sps_sensitivity["Ref.Freq.Width"]
    / sps_sensitivity["SPS Spec. Res."]
)
sps_sensitivity["reference_width_to_window_bandwidth_ratio"] = (
    sps_sensitivity["Ref.Freq.Width"]
    / sps_sensitivity["SPS Bandwidth"]
)

display(sps_sensitivity)

assert len(sps_sensitivity) == 1
assert sps_sensitivity["reference_inside_scan_range"].all()
assert sps_sensitivity[
    "reference_width_to_native_resolution_ratio"
].iloc[0] > 1000.0
assert sps_sensitivity[
    "reference_width_to_window_bandwidth_ratio"
].iloc[0] < 0.001

,Project Code,Target Name,Ref.Frequency,Ref.Freq.Width,Req.Sensitivity,SPS Start Freq.,SPS End Freq.,SPS Bandwidth,SPS Spec. Res.,reference_inside_scan_range,reference_width_to_native_resolution_ratio,reference_width_to_window_bandwidth_ratio
33,2025.1.00299.S,HBC_687,265.141,0.565,2.5,261.5,268.7,1000.0,0.000564,True,1000.968858,0.000565


### Sensitivity result

All 3,200 rows provide a positive sensitivity triple, but it is not constant within
every project–target–band group. This confirms that it belongs to the spectral/setup
side of reconstruction rather than a target-only record.

Using portal-script-derived usable SPW edges, 38 regular rows have a reference width
equal to the aggregate non-overlapping bandwidth, and only three match a raw SPW
spectral resolution. The remaining 3,158 widths are valid independent request values;
they may reflect a
user-defined smoothing width or another OT choice. The parser must preserve the
triple exactly and must not recalculate it from SPW columns.

For the one SPS row, the 265.141 GHz reference frequency lies inside the
261.5–268.7 GHz scan. Its 0.565 MHz reference width is about 1,001 times the
0.000564453125 MHz native scan resolution and only 0.000565 of the 1,000 MHz
per-window bandwidth. Therefore its 2.5 mJy request is defined at the separate
reference/smoothing width, not per native spectral element and not over a full scan
window.

`Req.Sensitivity` is requested proposal-side evidence. It is not an achieved Archive
sensitivity and must retain that provenance in the shared comparison model.

## 10. Direct correlator-mode evidence

The operational header has no named TDM/FDM or correlator-mode field. A literal-token
search is retained as a defensive check, but a token found in a target name or free
text would not automatically constitute authoritative per-SPW mode evidence.


In [26]:
mode_tokens = ["TDM", "FDM", "continuum", "line"]
mode_token_hits = []

for column in queue_raw.columns:
    values = queue_raw[column].astype(str)
    for token in mode_tokens:
        pattern = rf"(?i)(?<!\w){re.escape(token)}(?!\w)"
        hit_mask = values.str.contains(pattern, regex=True, na=False)

        if hit_mask.any():
            for row_index in queue_raw.index[hit_mask][:10]:
                mode_token_hits.append(
                    {
                        "token": token,
                        "column_name": column,
                        "source_line_number": int(source_line_numbers[row_index]),
                        "raw_value": queue_raw.at[row_index, column],
                    }
                )

mode_token_hits = pd.DataFrame(
    mode_token_hits,
    columns=[
        "token",
        "column_name",
        "source_line_number",
        "raw_value",
    ],
)

print("Mode-like operational columns:", [
    column
    for column in queue_raw.columns
    if re.search(r"(?i)mode|correlator|TDM|FDM", column)
])
display(mode_token_hits)


Mode-like operational columns: []


,token,column_name,source_line_number,raw_value


## 11. Closure decision and production parser boundary

### Evidence established for the pinned snapshot

- The physical file has 3,241 lines, including 35 embedded dictionary rows,
  79 operational columns, and 3,200 valid data rows.
- The raw table remains unchanged and every row has physical-line provenance.
- The schema reserves 16 complete, number-aligned SPW triples; the current snapshot
  uses slots 1–7 and contains 16,216 observed long-form SPW records.
- 3,199 rows use regular SPWs and one row uses a complete, mutually exclusive SPS
  representation.
- Seven explicit schema/unit drifts are recorded, including the conflicting SPS
  bandwidth unit. The dictionary's MHz interpretation is the only numerically
  plausible current-snapshot interpretation, but both declarations remain evidence.
- The file contains 419 project–target–band groups. Spatial and spectral signatures
  frequently form a complete local product, but two groups are sparse/paired and five
  groups contain exact export multiplicity. Relationships must come from raw rows.
- At the selected `1e-6 arcsec` tolerance, all 231 custom-mosaic groups reconstruct
  as one center plus six offset spatial components. Blank mosaic labels still include
  nine meaningful offsets.
- All 3,199 regular rows place `Ref.Frequency` inside at least one derived
  sky-frequency SPW interval. This validates CSV-side Doppler reconstruction while
  leaving Archive-frame alignment as a separate comparison concern.
- All 3,200 sensitivity triples are complete and positive. `Ref.Freq.Width` is an
  independent requested-sensitivity bandwidth, not generally SPW bandwidth or native
  resolution. The SPS row confirms the same separation.
- No queue column supplies authoritative per-SPW TDM/FDM evidence. A continuum policy
  criterion may later be computed from bandwidths, but it must not be relabelled as an
  authoritative correlator mode.

### Production model implied by the evidence

The production parser can now be designed around the following boundaries:

1. `QueueSnapshot`: source URL, checksum, capture metadata, description, embedded
   dictionary, raw unit row, and recorded schema discrepancies.
2. `RawQueueRow`: snapshot ID, physical line number, exact 79-column string mapping,
   and content fingerprint.
3. `QueueSpatialComponent`: coordinates, raw mosaic label, offsets, rectangle
   geometry, coordinate system, signature, and classification tolerance/provenance.
4. `QueueSpectralSetup`: a tagged union of `RegularSpwSetup` and `SpectralScanSetup`,
   including velocity/frame/convention, sky/rest declaration, the original
   sensitivity triple, normalized values, raw units, and derivation provenance.
5. `QueueRowAssociation`: the observed link from one raw row to one spatial
   component, one spectral setup, and one request context. It retains source lines and
   multiplicity; it never creates unobserved Cartesian pairs.
6. `QueueParseIssue`: explicit codes for schema drift, unit conflict, incomplete
   indexed triples, unsupported categories, and failed scientific consistency gates.

### Closure status

**Queue CSV exploration is complete for production parser v1.** The next step is to
write the parser contract and offline fixtures, not to continue broad notebook
exploration. Remaining policy questions—especially authoritative TDM/FDM evidence,
Archive/queue frame alignment, HPBW mosaic overlap, and sensitivity comparison after
smoothing—belong to the shared comparison and rule-engine phases.